<a href="https://colab.research.google.com/github/rounak393/clab/blob/main/HMU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

os.environ['KAGGLE_API_TOKEN'] = "KGAT_c03d989b55c966d18c971a92b023645b"

!kaggle datasets download -d britikak/busi-dataset

Dataset URL: https://www.kaggle.com/datasets/britikak/busi-dataset
License(s): unknown
100% 195M/195M [00:02<00:00, 90.6MB/s]



In [2]:

!unzip -q busi-dataset.zip -d busi_dataset

In [9]:
import os
import copy
import torch
import random
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
from PIL import Image
import albumentations as A
from tqdm import tqdm

torch.backends.cudnn.benchmark = True

BASE_DIR = "/content/busi_dataset/Dataset_BUSI_with_GT"
CLASSES = ["benign", "malignant"]
IMG_SIZE = 256
BATCH_SIZE = 8  # Strictly 8 to fit the standard heavy architecture in T4 VRAM
EPOCHS = 50

torch.manual_seed(100)
random.seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [4]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.net(x)

class DCSAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.smooth = nn.AvgPool2d(3, stride=1, padding=1)
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // 8, 4), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // 8, 4), channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.sa = nn.Sequential(nn.Conv2d(2, 1, 7, padding=3, bias=False), nn.Sigmoid())

    def forward(self, x):
        details = x - self.smooth(x)
        x = x + details
        x = x * self.ca(x)
        sa_in = torch.cat([torch.mean(x, dim=1, keepdim=True), torch.max(x, dim=1, keepdim=True)[0]], dim=1)
        return x * self.sa(sa_in)

class HMAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        inter = channels // 4
        self.reduce = nn.Conv2d(channels, inter * 3, 1, bias=False)
        self.d1 = nn.Conv2d(inter, inter, 3, padding=1, dilation=1, bias=False)
        self.d3 = nn.Conv2d(inter, inter, 3, padding=3, dilation=3, bias=False)
        self.d5 = nn.Conv2d(inter, inter, 3, padding=5, dilation=5, bias=False)

        self.pool = nn.AdaptiveAvgPool2d(1)
        self.pool_conv = nn.Conv2d(channels, inter, 1, bias=False)
        self.fuse = nn.Conv2d(inter * 4, channels, 1, bias=False)
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // 8, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // 8, channels, 1, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        b1, b2, b3 = torch.split(self.reduce(x), x.shape[1] // 4, dim=1)
        b1, b2, b3 = self.d1(b1), self.d3(b2), self.d5(b3)
        b4 = F.interpolate(self.pool_conv(self.pool(x)), size=x.shape[2:], mode='bilinear', align_corners=False)
        fused = self.fuse(torch.cat([b1, b2, b3, b4], dim=1))
        return fused * self.se(fused)

In [5]:
class HMUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        self.enc1 = DoubleConv(in_channels, 64)
        self.enc2 = DoubleConv(64, 128)
        self.enc3 = DoubleConv(128, 256)
        self.enc4 = DoubleConv(256, 512)
        self.pool = nn.MaxPool2d(2)

        self.dcsam1 = DCSAM(64)
        self.dcsam2 = DCSAM(128)
        self.dcsam3 = DCSAM(256)
        self.dcsam4 = DCSAM(512)

        self.bottleneck = DoubleConv(512, 1024)
        self.hmam = HMAM(1024)

        self.up4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(1024, 512)
        self.up3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(512, 256)
        self.up2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(256, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(128, 64)

        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))

        b = self.hmam(self.bottleneck(self.pool(e4)))

        d4 = self.dec4(torch.cat([self.dcsam4(e4), self.up4(b)], dim=1))
        d3 = self.dec3(torch.cat([self.dcsam3(e3), self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([self.dcsam2(e2), self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([self.dcsam1(e1), self.up1(d2)], dim=1))
        return self.final_conv(d1)

In [10]:
model = HMUNet(in_channels=3, out_channels=1).to(device)
criterion = StrictBCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

scaler = torch.amp.GradScaler('cuda')

best_val_dice = 0.0
best_model_weights = None

for epoch in range(EPOCHS):
    # 1. TRAINING PHASE
    model.train()
    train_loss = 0
    train_dice = 0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()


        with torch.no_grad():
            train_dice += strict_dice_coef(masks, logits).item()

    scheduler.step()

    avg_train_loss = train_loss / len(train_loader)
    avg_train_dice = train_dice / len(train_loader)

    # 2. VALIDATION PHASE
    model.eval()
    val_loss = val_dice = 0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, masks)

            val_loss += loss.item()
            val_dice += strict_dice_coef(masks, logits).item()

    avg_val_loss = val_loss / len(val_loader)
    avg_val_dice = val_dice / len(val_loader)

    print(f"Train Loss: {avg_train_loss:.4f} | Train Dice: {avg_train_dice:.4f} || Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")

    # SAVE BEST MODEL (Based on Validation)
    if avg_val_dice > best_val_dice:
        best_val_dice = avg_val_dice
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), "best_hmunet.pth")


# 3. TEST PHASE
print("\n--- Training Complete ---")
model.load_state_dict(best_model_weights)
model.eval()
test_loss = test_dice = 0

with torch.no_grad():
    for images, masks in test_loader:
        images, masks = images.to(device, non_blocking=True), masks.to(device, non_blocking=True)

        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, masks)

        test_loss += loss.item()
        test_dice += strict_dice_coef(masks, logits).item()

avg_test_loss = test_loss / len(test_loader)
avg_test_dice = test_dice / len(test_loader)

print(f" Final Test Loss: {avg_test_loss:.4f} | Final Test Dice: {avg_test_dice:.4f}")

Epoch 1/50: 100%|██████████| 65/65 [00:17<00:00,  3.72it/s]


Train Loss: 0.7567 | Train Dice: 0.2763 || Val Loss: 0.7442 | Val Dice: 0.3122


Epoch 2/50: 100%|██████████| 65/65 [00:17<00:00,  3.73it/s]


Train Loss: 0.6795 | Train Dice: 0.4189 || Val Loss: 0.7309 | Val Dice: 0.3912


Epoch 3/50: 100%|██████████| 65/65 [00:17<00:00,  3.68it/s]


...

Train Loss: 0.2186 | Train Dice: 0.7804 || Val Loss: 0.1937 | Val Dice: 0.8038


Epoch 49/50: 100%|██████████| 65/65 [00:17<00:00,  3.76it/s]


Train Loss: 0.2037 | Train Dice: 0.7989 || Val Loss: 0.1913 | Val Dice: 0.8064


Epoch 50/50: 100%|██████████| 65/65 [00:17<00:00,  3.76it/s]


Train Loss: 0.2146 | Train Dice: 0.7863 || Val Loss: 0.1918 | Val Dice: 0.8060

--- Training Complete ---
 Final Test Loss: 0.3518 | Final Test Dice: 0.6242
